Separate NSL-KDD interpretability exploration using training-only encoding and scaling. Benchmark difficulty metadata is excluded from predictors. Outputs are cleared for a fresh run. Feature importance describes model associations.


In [ ]:
# Install dependencies from requirements.txt before running.

In [ ]:
# Install dependencies from requirements.txt before running.

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import shap
import lime
import lime.lime_tabular
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load the NSL-KDD dataset
# Download NSL-KDD train dataset from: https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt
url = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt'
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

df = pd.read_csv(url, header=None, names=columns)

In [ ]:
df

In [ ]:
# Print the categorical columns in the dataset
cat_df = df.select_dtypes(include=['object'])
print(cat_df.columns)

In [ ]:
# Hold out data before learning categorical encodings or scaling.
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

y = (df['label'] != 'normal').astype(int)
X_raw = df.drop(columns=['label', 'difficulty'])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=42)
categorical = ['protocol_type', 'service', 'flag']
numeric = [c for c in X_raw.columns if c not in categorical]
preprocessor = ColumnTransformer([
    ('categorical', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical),
    ('numeric', MinMaxScaler(), numeric)])
feature_names = categorical + numeric
X_train = pd.DataFrame(preprocessor.fit_transform(X_train_raw), columns=feature_names, index=X_train_raw.index)
X_test = pd.DataFrame(preprocessor.transform(X_test_raw), columns=feature_names, index=X_test_raw.index)
X = X_train


In [ ]:
df['label'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
df['label']

In [ ]:
# Split and preprocessing are defined above using training data only.

In [ ]:
df

In [ ]:
# Split and preprocessing are defined above using training data only.

In [ ]:
X_train.head()

In [ ]:
# Split and preprocessing are defined above using training data only.

In [ ]:
# Train a RandomForest model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Feature Importance (Tree-based)
importances = model.feature_importances_
feature_names = X.columns

In [ ]:
# Permutation Importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

In [ ]:
# Partial Dependence Plot
PartialDependenceDisplay.from_estimator(model, X_train, features=[0], feature_names=feature_names)
plt.title('Partial Dependence Plot')
plt.show()

# Partial Dependence Plot (PDP)
PDPs show the marginal effect one or two features have on the predicted outcome of a machine learning model. It helps to understand how the prediction changes when a feature's value changes.


1.   The PDP plot shows how changes in the duration feature (feature index 0) affect the model's predictions.
2.   It shows the average prediction response as duration varies, holding all other features constant.
3.   A flat line would indicate the feature has little effect, while a varying line shows significant influence.




In [ ]:
# SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
elif np.asarray(shap_values).ndim == 3:
    shap_values = shap_values[:, :, 1]
shap.summary_plot(shap_values, X_train)

# SHAP Values
SHAP (SHapley Additive exPlanations) values provide a unified measure of feature importance and individual prediction explanations. They are based on cooperative game theory and offer both global and local interpretability.
1. The summary plot shows the impact of each feature on the model's output.
2. Each dot represents a SHAP value for a feature and a single instance.
3. The color represents the feature value (e.g., red for high values, blue for low values).
4. The position on the X-axis shows whether the effect of that value is associated with a higher or lower prediction.



In [ ]:
# LIME
explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values, feature_names=feature_names, class_names=['normal', 'attack'], mode='classification')
exp = explainer.explain_instance(X_train.iloc[0].values, model.predict_proba, num_features=10)
exp.show_in_notebook(show_all=False)

# LIME
LIME (Local Interpretable Model-agnostic Explanations) provides local interpretability by approximating the model locally with an interpretable model like linear regression.
1. LIME explains the prediction of a single instance (first instance in this case).
2. It shows which features contributed most to the specific prediction.
3. This helps in understanding and debugging individual predictions.

In [ ]:
# Correlation Matrix
corr_matrix = X_train.corr()
plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=True, cmap=plt.cm.Reds)
plt.title('Correlation Matrix')
plt.show()

# Correlation Matrix
The correlation matrix shows the pairwise correlation coefficients between features. This can reveal multicollinearity and relationships between features.
1. The heatmap shows correlation coefficients between features.
2. Values close to 1 or -1 indicate strong correlation, while values close to 0 indicate weak correlation.
3. Positive values indicate positive correlation, and negative values indicate negative correlation.



In [ ]:
# Feature Selection
selector = SelectKBest(f_classif, k=10)
selector.fit(X_train, y_train)
selected_features = selector.get_support(indices=True)
selected_feature_names = [feature_names[i] for i in selected_features]
print('Selected features:', selected_feature_names)

# Feature Selection
SelectKBest selects the top k features based on statistical tests (ANOVA F-test in this case).
1. This output lists the top 10 features selected by the ANOVA F-test.
2. These features have the strongest relationship with the target variable based on the test.

In [ ]:
# Visualizations
plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=feature_names)
plt.title('Feature Importances from Random Forest')
plt.show()

# Feature Importance (Tree-based)
Tree-based feature importance provides a way to understand which features are most influential in making predictions. The RandomForest model computes this by looking at how much each feature reduces the impurity (e.g., Gini impurity) in the trees of the forest.

1.   The bar plot shows the importance of each feature. Higher bars indicate features that contribute more to the model's decisions.
2.   Features with higher importance scores have a larger impact on the model’s prediction.



In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=perm_importance.importances_mean, y=feature_names)
plt.title('Permutation Importances')
plt.show()

# Permutation Importance
Permutation importance measures the change in the model's prediction error when the values of a feature are randomly shuffled. This process breaks the relationship between the feature and the target, and the increase in error indicates the importance of the feature.


1.   This bar plot shows the mean importance of each feature based on multiple permutations.
2.   Features with higher permutation importance scores indicate they are more critical for accurate predictions.

